In [1]:
import warnings
import pandas as pd
import os

# Suppress all FutureWarnings
warnings.simplefilter(action='ignore', category=FutureWarning)

def format_duration(duration):
    seconds = duration.total_seconds()
    hours = int(seconds // 3600)
    minutes = int((seconds % 3600) // 60)
    seconds = int(seconds % 60)
    return f"{hours:02}:{minutes:02}:{seconds:02}"

def check_tp_sl(price_data, entry_datetime, tp_price, sl_price, side):
    result = 0
    exit_datetime = None
    
    # Filter the subsequent prices correctly (inclusive of the entry_datetime)
    subsequent_prices = price_data.loc[entry_datetime:]

    for current_datetime, price_row in subsequent_prices.iterrows():
        if side == 'Buy':
            if price_row['High'] >= tp_price:
                result = 1
                exit_datetime = current_datetime
                break
            elif price_row['Low'] <= sl_price:
                result = -1
                exit_datetime = current_datetime
                break
        else:
            if price_row['Low'] <= tp_price:
                result = 1
                exit_datetime = current_datetime
                break
            elif price_row['High'] >= sl_price:
                result = -1
                exit_datetime = current_datetime
                break
    
    if exit_datetime:
        duration = exit_datetime - entry_datetime
        duration_str = format_duration(duration)
    else:
        duration_str = '00:00:00'
        
    return result, duration_str

def determine_entry_by_percentage(price_data, signal_datetime, percentage_change, side, time_limit_minutes):
    signal_open_price = price_data.at[signal_datetime, 'Open']
    percentage_change_price = signal_open_price * (1 - percentage_change) if side == 'Buy' else signal_open_price * (1 + percentage_change)
    
    time_limit = signal_datetime + pd.Timedelta(minutes=time_limit_minutes)
    subsequent_prices = price_data.loc[signal_datetime:time_limit]
    
    for current_datetime, price_row in subsequent_prices.iterrows():
        if side == 'Buy' and price_row['Low'] <= percentage_change_price:
            duration = current_datetime - signal_datetime
            return current_datetime, percentage_change_price, duration
        elif side == 'Sell' and price_row['High'] >= percentage_change_price:
            duration = current_datetime - signal_datetime
            return current_datetime, percentage_change_price, duration

    return None, None, None

def backtest_trades(price_data, signal_data, tp=None, sl=None, entry_time_offset=None, percentage_change=None, entry_method=None, time_limit_minutes=None):
    output_data = pd.DataFrame(columns=[
        'Datetime', 'Side', 'Signal Open Price', 'Entry Price', 'TP Price', 'SL Price', 'Result', 'Duration', 'Execution Latency', 'ROI', 'NAV'
    ])
    
    initial_margin = 100000
    current_margin = initial_margin

    for i, row in signal_data.iterrows():
        signal_datetime = row['Datetime']
        signal_value = row['Signal']
        
        if signal_value == 0:
            continue
        elif signal_value > 0:
            side = 'Buy'
        else:
            side = 'Sell'
        
        signal_open_price = price_data.at[signal_datetime, 'Open']
        
        if entry_method == 'percentage_change':
            entry_datetime, entry_price, entry_duration = determine_entry_by_percentage(price_data, signal_datetime, percentage_change, side, time_limit_minutes)
            if entry_datetime is None:
                new_row = pd.DataFrame([{
                    'Datetime': signal_datetime,
                    'Side': side,
                    'Signal Open Price': signal_open_price,
                    'Entry Price': None,
                    'TP Price': None,
                    'SL Price': None,
                    'Result': 'Not Filled',
                    'Duration': '00:00:00',
                    'Execution Latency': '00:00:00',
                    'ROI': 0,
                    'NAV': current_margin
                }])
                output_data = pd.concat([output_data, new_row], ignore_index=True)
                continue
        elif entry_method == 'time_offset':
            entry_datetime = signal_datetime + pd.Timedelta(minutes=entry_time_offset)
            if entry_datetime not in price_data.index:
                continue
            entry_price = price_data.at[entry_datetime, 'Open']
            entry_duration = entry_datetime - signal_datetime
        else:
            continue
        
        if side == 'Buy':
            tp_price = entry_price * (1 + tp)
            sl_price = entry_price * (1 - sl)
        else:
            tp_price = entry_price * (1 - tp)
            sl_price = entry_price * (1 + sl)
        
        result, duration_str = check_tp_sl(price_data, entry_datetime, tp_price, sl_price, side)
       
        if result == 1:
            current_margin = current_margin * (1 + tp)
        elif result == -1:
            current_margin = current_margin * (1 - sl)
        
        roi = ((current_margin - initial_margin) / initial_margin) * 100
        nav = current_margin
        initial_margin = current_margin
        
        new_row = pd.DataFrame([{
            'Datetime': signal_datetime,
            'Side': side,
            'Signal Open Price': signal_open_price,
            'Entry Price': entry_price,
            'TP Price': tp_price,
            'SL Price': sl_price,
            'Result': result,
            'Duration': duration_str,
            'Execution Latency': format_duration(entry_duration),
            'ROI': roi,
            'NAV': nav
        }])
        
        output_data = pd.concat([output_data, new_row], ignore_index=True)
    
    return output_data

def calculate_metrics(group, initial_nav):
    total_trades = len(group)
    total_wins = len(group[group['Result'] == 1])
    total_losses = len(group[group['Result'] == -1])
    win_rate = total_wins / total_trades if total_trades > 0 else 0
    
    final_nav = group['NAV'].iloc[-1] if total_trades > 0 else initial_nav
    roi = ((final_nav - initial_nav) / initial_nav) * 100
    
    drawdown = 0
    cumulative_returns = (group['NAV'] - initial_nav).cumsum()
    peak = cumulative_returns.cummax()
    drawdown = (peak - cumulative_returns).max()
    
    return {
        'Total Trades': total_trades,
        'Total Wins': total_wins,
        'Total Losses': total_losses,
        'Win Rate': win_rate,
        'ROI': roi,
        'NAV': final_nav,
        'Max Drawdown': drawdown
    }

def generate_report(price_data, signal_data, scenarios, output_directory):
    report_columns = ['Scenario', 'Period', 'Total Trades', 'Total Wins', 'Total Losses', 'Win Rate', 'ROI', 'NAV', 'Max Drawdown']
    report_data = pd.DataFrame(columns=report_columns)

    for scenario in scenarios:
        tp = scenario['tp']
        sl = scenario['sl']
        entry_method = scenario['entry_method']
        percentage_change = scenario.get('percentage_change', None)
        entry_time_offset = scenario.get('entry_time_offset', None)
        time_limit_minutes = scenario.get('time_limit_minutes', None)
        
        # Backtest the trades for the current scenario
        trade_data = backtest_trades(price_data, signal_data, tp, sl, entry_time_offset, percentage_change, entry_method, time_limit_minutes)
        
        # Calculate metrics for each month
        monthly_groups = trade_data.groupby(trade_data['Datetime'].dt.to_period('M'))
        monthly_reports = []
        initial_nav = 100000
        for month, group in monthly_groups:
            metrics = calculate_metrics(group, initial_nav)
            metrics['Period'] = month.strftime('%Y-%m')
            metrics['Scenario'] = f"TP={tp}, SL={sl}, Method={entry_method}, pctChange={percentage_change}, TimeLimit={time_limit_minutes}"
            monthly_reports.append(pd.DataFrame([metrics]))
            initial_nav = metrics['NAV']  # Update the initial_nav for the next interval

        if monthly_reports:
            monthly_report = pd.concat(monthly_reports, ignore_index=True)
            report_data = pd.concat([report_data, monthly_report], ignore_index=True)

        # Calculate metrics for the overall period
        overall_metrics = calculate_metrics(trade_data, 100000)
        overall_metrics['Period'] = 'Overall'
        overall_metrics['Scenario'] = f"TP={tp}, SL={sl}, Method={entry_method}, pctChange={percentage_change}, TimeLimit={time_limit_minutes}"
        overall_report = pd.DataFrame([overall_metrics])
        report_data = pd.concat([report_data, overall_report], ignore_index=True)

    # Ensure the output directory exists
    os.makedirs(output_directory, exist_ok=True)
    
    # Save the compiled report
    report_data.to_csv(os.path.join(output_directory, '80_backtesting.csv'), index=False)

    return report_data

In [2]:
# Load the price data
price_data = pd.read_csv('E:\Signal Backtesting\Input\Price_2024.csv')
price_data['Datetime'] = pd.to_datetime(price_data['Datetime'])
price_data.set_index('Datetime', inplace=True)
price_data.sort_index(ascending=True, inplace=True)

signal_data = pd.read_csv('E:\Signal Backtesting\Output\\178\\updated_dataset.csv')
signal_data['Datetime'] = pd.to_datetime(signal_data['Datetime'])

In [4]:
result=backtest_trades(price_data, signal_data, tp=0.0096, sl=0.014, entry_time_offset=0,time_limit_minutes=120 ,percentage_change=0.00008, entry_method='percentage_change')
# result.to_csv(os.path.join('E:\Signal Backtesting\Output\\178\optimized parameters with leverages', 'result_178_tp=0.96_sl=1.38_pct=8.csv'), index=False)
result

,Datetime,Side,Signal Open Price,Entry Price,TP Price,SL Price,Result,Duration,Execution Latency,ROI,NAV
0,2024-01-01 02:58:00,Sell,42623.7,42627.109896,42217.889641,43223.889435,1,02:54:00,00:04:00,0.96,100960.000000
1,2024-01-01 18:58:00,Buy,43117.4,NaN,NaN,NaN,Not Filled,00:00:00,00:00:00,0.00,100960.000000
2,2024-01-02 02:58:00,Sell,45376.4,45380.030112,44944.381823,46015.350534,1,12:19:00,00:00:00,0.96,101929.216000
3,2024-01-03 00:58:00,Buy,45148.9,45145.288088,45578.682854,44513.254055,1,08:38:00,00:03:00,0.96,102907.736474
4,2024-01-03 18:58:00,Buy,42248.0,42244.620160,42650.168514,41653.195478,1,00:24:00,00:00:00,0.96,103895.650744
...,...,...,...,...,...,...,...,...,...,...,...
214,2024-06-27 10:58:00,Sell,61125.0,61129.890000,60543.043056,61985.708460,-1,02:59:00,00:02:00,-1.40,147533.035961
215,2024-06-28 10:58:00,Buy,61550.1,61545.175992,62136.009682,60683.543528,-1,05:43:00,00:00:00,-1.40,145467.573457
216,2024-06-28 18:58:00,Sell,60837.3,60842.166984,60258.082181,61693.957322,1,00:17:00,00:02:00,0.96,146864.062162
217,2024-06-29 05:58:00,Buy,60775.0,60770.138000,61353.531325,59919.356068,1,25:06:00,00:01:00,0.96,148273.957159


In [4]:
scenarios = [
      {'tp': 0.0096, 'sl': 0.014, 'entry_method': 'time_offset', 'entry_time_offset': 80},
#      {'tp': 0.0092, 'sl': 0.016, 'entry_method': 'time_offset', 'entry_time_offset': 178},
#      {'tp': 0.0093, 'sl': 0.016, 'entry_method': 'time_offset', 'entry_time_offset': 178},
#      {'tp': 0.0094, 'sl': 0.016, 'entry_method': 'time_offset', 'entry_time_offset': 178},
#      {'tp': 0.0095, 'sl': 0.016, 'entry_method': 'time_offset', 'entry_time_offset': 178},
#      {'tp': 0.0096, 'sl': 0.016, 'entry_method': 'time_offset', 'entry_time_offset': 178},
#     {'tp': 0.0097, 'sl': 0.016, 'entry_method': 'time_offset', 'entry_time_offset': 178},
#     {'tp': 0.0098, 'sl': 0.016, 'entry_method': 'time_offset', 'entry_time_offset': 178},
#     {'tp': 0.0099, 'sl': 0.016, 'entry_method': 'time_offset', 'entry_time_offset': 178},
#     {'tp': 0.01, 'sl': 0.016, 'entry_method': 'time_offset', 'entry_time_offset': 178},
#     {'tp': 0.011, 'sl': 0.016, 'entry_method': 'time_offset', 'entry_time_offset': 178},
#     {'tp': 0.012, 'sl': 0.016, 'entry_method': 'time_offset', 'entry_time_offset': 178},
#     {'tp': 0.013, 'sl': 0.016, 'entry_method': 'time_offset', 'entry_time_offset': 178},
#     {'tp': 0.014, 'sl': 0.016, 'entry_method': 'time_offset', 'entry_time_offset': 178},
#     {'tp': 0.015, 'sl': 0.016, 'entry_method': 'time_offset', 'entry_time_offset': 178}
#        {'tp': 0.0096, 'sl': 0.0139, 'entry_method': 'percentage_change', 'percentage_change': 0.00008,'time_limit_minutes': 100000000}, 
#{'tp': 0.0096, 'sl': 0.0096, 'entry_method': 'percentage_change', 'percentage_change': 0.00008,'time_limit_minutes': 120} 
#  {'tp': 0.0096, 'sl': 0.014, 'entry_method': 'percentage_change', 'percentage_change': 0.0001,'time_limit_minutes': 120},
#  {'tp': 0.0096, 'sl': 0.014, 'entry_method': 'percentage_change', 'percentage_change': 0.00011,'time_limit_minutes': 120},
# {'tp': 0.0096, 'sl': 0.014, 'entry_method': 'percentage_change', 'percentage_change': 0.00012,'time_limit_minutes': 120}, {'tp': 0.0096, 'sl': 0.014, 'entry_method': 'percentage_change', 'percentage_change': 0.00013,'time_limit_minutes': 120}
]
report = generate_report(price_data, signal_data, scenarios, output_directory='E:\Signal Backtesting\Output')